# Step 2 : Event Data Preprocessing

# Importing the nessasary libraries

In [4]:
# Importing the nessasary libraries
import numpy as np
import pandas as pd
import json  # for reading the .json neural data file
import os

# Importing the data

In [6]:
# Defining the data paths
NEURAL_DATA_PATH = r"C:\Users\ASSUS\ATN\Digit Span Backwards\Data\Neural Data\DBS ATN DSB Case 1\D. Siragusa\3.5.26\Time stamp 1433\Report_Json_Session_Report_20260305T151703.json"
EPRIME_DATA_PATH = r"C:\Users\ASSUS\ATN\Digit Span Backwards\Data\Eprime Data\Digit Span Backwards v3.2\DigitSpanBackward v3.3-6-2-Scores.csv"

# ── Sync constants (from Code_Adaptation_Guide) ──────────────────────────
# The first DBS stimulation spike in the neural recording anchors t=0.
# neural_sample = (RTTime_ms - FORMULA_CONSTANT) / MS_PER_SAMPLE
FORMULA_CONSTANT = 1199133   # ms – clock offset verified from first DBS stim spike
MS_PER_SAMPLE    = 4.0     # 1000 / 250 Hz
SAMPLING_RATE    = 250     # Hz

def rttime_to_sample(rttime_ms):
    """Convert E-Prime RTTime (ms) to neural sample index."""
    return int(round((rttime_ms - FORMULA_CONSTANT) / MS_PER_SAMPLE))

In [7]:
# Importing the neural data
with open(NEURAL_DATA_PATH, 'r') as f:
    data = json.load(f)

# Quick check
[(entry['Channel'], entry['SampleRateInHz'], len(entry['TimeDomainData']))
 for entry in data['BrainSenseTimeDomain']]

[('ZERO_THREE_LEFT', 250, 23750),
 ('ZERO_THREE_RIGHT', 250, 23750),
 ('ZERO_THREE_LEFT', 250, 16313),
 ('ZERO_THREE_RIGHT', 250, 16313),
 ('ZERO_THREE_LEFT', 250, 7187),
 ('ZERO_THREE_RIGHT', 250, 7187),
 ('ZERO_THREE_LEFT', 250, 750),
 ('ZERO_THREE_RIGHT', 250, 750),
 ('ZERO_THREE_LEFT', 250, 563),
 ('ZERO_THREE_RIGHT', 250, 563),
 ('ZERO_THREE_LEFT', 250, 375),
 ('ZERO_THREE_RIGHT', 250, 375)]

In [8]:
# No separate marker list needed – events come from the E-Prime scores CSV.
# The sync anchor is defined by FORMULA_CONSTANT above.
n_samples = len(data['BrainSenseTimeDomain'][0]['TimeDomainData'])
print(f'Neural data: {n_samples} samples at {SAMPLING_RATE} Hz')

Neural data: 23750 samples at 250 Hz


## Seperating event logs

### Creating a dataframe for the event markers

In [9]:
eprime_raw = pd.read_csv(EPRIME_DATA_PATH)

rows = []

# Session Start (event 6) — taken from the first row's Welcome.TargetOnsetTime
session_start_ms = eprime_raw['Welcome.TargetOnsetTime'].iloc[0]
rows.append({'Time_ms': session_start_ms, 'Event': 6.0, 'Block': np.nan, 'Trial_In_Block': np.nan})

# Session End (event 7) — taken from the first row's Goodbye.OffsetTime
session_end_ms = eprime_raw['Goodbye.OffsetTime'].iloc[0]
rows.append({'Time_ms': session_end_ms, 'Event': 7.0, 'Block': np.nan, 'Trial_In_Block': np.nan})

for (block, trial), grp in eprime_raw.groupby(['Block', 'Trial'], sort=False):
    first = grp.iloc[0]
    last  = grp.iloc[-1]

    # Main Trial Start — same time as first Fixation
    rows.append({'Time_ms': first['Fixation.TargetOnsetTime'],
                 'Event': 8.0, 'Block': block, 'Trial_In_Block': trial})

    # Per subtrial: Fixation → Stimulus (interleaved)
    for _, row in grp.iterrows():
        rows.append({'Time_ms': row['Fixation.TargetOnsetTime'],
                     'Event': 1.0, 'Block': block, 'Trial_In_Block': trial})
        rows.append({'Time_ms': row['Fixation.OffsetTime'],
                     'Event': 2.0, 'Block': block, 'Trial_In_Block': trial})
        rows.append({'Time_ms': row['Stimulus.TargetOnsetTime'],
                     'Event': 10.0, 'Block': block, 'Trial_In_Block': trial})
        rows.append({'Time_ms': row['Stimulus.OffsetTime'],
                     'Event': 11.0, 'Block': block, 'Trial_In_Block': trial})

    # Choice Start / End
    rows.append({'Time_ms': first['CollectResponse.TargetOnsetTime'],
                 'Event': 12.0, 'Block': block, 'Trial_In_Block': trial})
    rows.append({'Time_ms': first['CollectResponse.OffsetTime'],
                 'Event': 13.0, 'Block': block, 'Trial_In_Block': trial})

    # Feedback Start / End
    rows.append({'Time_ms': first['Feedback.TargetOnsetTime'],
                 'Event': 14.0, 'Block': block, 'Trial_In_Block': trial})
    rows.append({'Time_ms': first['Feedback.OffsetTime'],
                 'Event': 15.0, 'Block': block, 'Trial_In_Block': trial})

    # Main Trial End — last subtrial's Feedback offset
    rows.append({'Time_ms': last['Feedback.OffsetTime'],
                 'Event': 9.0, 'Block': block, 'Trial_In_Block': trial})

event_order = {6.0: 0, 8.0: 1, 1.0: 2, 2.0: 3, 10.0: 4, 11.0: 5,
               12.0: 6, 13.0: 7, 14.0: 8, 15.0: 9, 9.0: 10, 7.0: 11}

for i, row in enumerate(rows):
    row['_insert_order'] = i
    row['_sort_key'] = event_order.get(row['Event'], 99)

events_data = (pd.DataFrame(rows)
               .sort_values(['Time_ms', '_insert_order', '_sort_key'])
               .drop(columns=['_insert_order', '_sort_key'])
               .reset_index(drop=True))

events_data['Time'] = events_data['Time_ms'].apply(rttime_to_sample) / SAMPLING_RATE
events_data = events_data[['Time', 'Time_ms', 'Event', 'Block', 'Trial_In_Block']]
events_data

,Time,Time_ms,Event,Block,Trial_In_Block
0,-1165.768,33367,6.0,NaN,NaN
1,-1131.700,67434,8.0,1.0,1.0
2,-1131.700,67434,1.0,1.0,1.0
3,-1131.684,67448,2.0,1.0,1.0
4,-1130.688,68447,10.0,1.0,1.0
...,...,...,...,...,...
281,-972.024,227109,13.0,7.0,2.0
282,-972.024,227109,14.0,7.0,2.0
283,-970.516,228618,15.0,7.0,2.0
284,-970.516,228618,9.0,7.0,2.0


### Adding labels to event markers

In [10]:
event_mapping = {
    6.0: 'Session Start',
    7.0: 'Session End',
    8.0: 'Main Trial Start',
    9.0: 'Main Trial End',
    1.0: 'Fixation Start',
    2.0: 'Fixation End',
    10.0: 'Stimulus Start',
    11.0: 'Stimulus End',
    12.0: 'Choice Start',
    13.0: 'Choice End',
    14.0: 'Feedback Start',
    15.0: 'Feedback End',
}

events_data['Event_Type'] = events_data['Event'].map(event_mapping)
events_data

,Time,Time_ms,Event,Block,Trial_In_Block,Event_Type
0,-1165.768,33367,6.0,NaN,NaN,Session Start
1,-1131.700,67434,8.0,1.0,1.0,Main Trial Start
2,-1131.700,67434,1.0,1.0,1.0,Fixation Start
3,-1131.684,67448,2.0,1.0,1.0,Fixation End
4,-1130.688,68447,10.0,1.0,1.0,Stimulus Start
...,...,...,...,...,...,...
281,-972.024,227109,13.0,7.0,2.0,Choice End
282,-972.024,227109,14.0,7.0,2.0,Feedback Start
283,-970.516,228618,15.0,7.0,2.0,Feedback End
284,-970.516,228618,9.0,7.0,2.0,Main Trial End


In [11]:
print("\nEvent type counts:")
print(events_data['Event_Type'].value_counts())


Event type counts:
Event_Type
Fixation Start      50
Fixation End        50
Stimulus Start      50
Stimulus End        50
Main Trial Start    14
Choice Start        14
Choice End          14
Feedback Start      14
Feedback End        14
Main Trial End      14
Session Start        1
Session End          1
Name: count, dtype: int64


### Adding trial number labels

In [12]:
def add_trial_numbers(events_df):
    """
    Add Trial_Number column where trials are defined as:
    - Start: Main Trial Start (Event 8)
    - End: Main Trial End (Event 9)
    
    All events between a trial start and end get the same trial number.
    Events outside trials get NaN.
    """
    events_with_trials = events_df.copy()
    events_with_trials['Trial_Number'] = np.nan
    
    current_trial = 0
    in_trial = False
    
    for idx, row in events_with_trials.iterrows():
        if row['Event'] == 8.0:  # Main Trial Start
            current_trial += 1
            in_trial = True
            events_with_trials.loc[idx, 'Trial_Number'] = current_trial
        elif row['Event'] == 9.0:  # Main Trial End
            if in_trial:
                events_with_trials.loc[idx, 'Trial_Number'] = current_trial
                in_trial = False
        else:
            if in_trial:
                events_with_trials.loc[idx, 'Trial_Number'] = current_trial
    
    # Convert to int where not NaN
    events_with_trials['Trial_Number'] = events_with_trials['Trial_Number'].astype('Int64')
    
    print(f"Found {current_trial} trials")
    print(f"Events within trials: {events_with_trials['Trial_Number'].notna().sum()}")
    print(f"Events outside trials: {events_with_trials['Trial_Number'].isna().sum()}")
    
    return events_with_trials

In [14]:
events_data = add_trial_numbers(events_data)
events_data

Found 14 trials
Events within trials: 284
Events outside trials: 2


,Time,Time_ms,Event,Block,Trial_In_Block,Event_Type,Trial_Number
0,-1165.768,33367,6.0,NaN,NaN,Session Start,<NA>
1,-1131.700,67434,8.0,1.0,1.0,Main Trial Start,1
2,-1131.700,67434,1.0,1.0,1.0,Fixation Start,1
3,-1131.684,67448,2.0,1.0,1.0,Fixation End,1
4,-1130.688,68447,10.0,1.0,1.0,Stimulus Start,1
...,...,...,...,...,...,...,...
281,-972.024,227109,13.0,7.0,2.0,Choice End,14
282,-972.024,227109,14.0,7.0,2.0,Feedback Start,14
283,-970.516,228618,15.0,7.0,2.0,Feedback End,14
284,-970.516,228618,9.0,7.0,2.0,Main Trial End,14


### Aligning the events with neural recordings

In [141]:
"""
SYNC METHOD (custom DBS data)
│
│  E-Prime clock (ms) → Neural sample index
│
│  neural_sample = (RTTime_ms - FORMULA_CONSTANT) / MS_PER_SAMPLE
│
│  FORMULA_CONSTANT = 73,455 ms
│      Verified from first DBS stimulation spike in the neural recording.
│      The first stimulation is where the task was initiated (t=0 anchor).
│
│  MS_PER_SAMPLE = 4.0  (1000 / 250 Hz)
│
│  Neural sample range: 0 – 69,999  (70,000 samples at 250 Hz)
"""

'\nSYNC METHOD (custom DBS data)\n│\n│  E-Prime clock (ms) → Neural sample index\n│\n│  neural_sample = (RTTime_ms - FORMULA_CONSTANT) / MS_PER_SAMPLE\n│\n│  FORMULA_CONSTANT = 73,455 ms\n│      Verified from first DBS stimulation spike in the neural recording.\n│      The first stimulation is where the task was initiated (t=0 anchor).\n│\n│  MS_PER_SAMPLE = 4.0  (1000 / 250 Hz)\n│\n│  Neural sample range: 0 – 69,999  (70,000 samples at 250 Hz)\n'

In [15]:
def align_events_to_neural_data(events_df, sampling_rate,
                                neural_data_start_time,
                                n_samples):
    """
    Align event timestamps to neural data sample indices.

    For our custom data, Time_ms is the E-Prime clock time and the conversion
    formula is: Sample_Index = (Time_ms - FORMULA_CONSTANT) / MS_PER_SAMPLE
    Time_Relative = Sample_Index / sampling_rate

    Parameters:
    -----------
    events_df : pandas.DataFrame
        DataFrame with 'Time_ms' column (E-Prime RTTime in ms)
    sampling_rate : float
        Neural data sampling rate in Hz (250)
    neural_data_start_time : float
        Unused for custom data (kept for API compatibility)
    n_samples : int
        Total number of samples in neural data (70000)

    Returns:
    --------
    events_df : pandas.DataFrame
        Original DataFrame with added columns:
        - 'Sample_Index': Sample index in neural data (0-based)
        - 'Time_Relative': Time relative to recording start (seconds)
        - 'Valid': Whether event falls within recording period
    """

    events_aligned = events_df.copy()

    # Convert E-Prime Time_ms to neural sample index using the sync formula
    events_aligned['Sample_Index'] = events_aligned['Time_ms'].apply(
        lambda t: int(round((t - FORMULA_CONSTANT) / MS_PER_SAMPLE))
    )

    # Time relative to neural recording start
    events_aligned['Time_Relative'] = events_aligned['Sample_Index'] / sampling_rate

    # Mark valid events (within neural data bounds)
    events_aligned['Valid'] = (
        (events_aligned['Sample_Index'] >= 0) &
        (events_aligned['Sample_Index'] < n_samples)
    )

    # Recording end time
    recording_duration = n_samples / sampling_rate

    # Print summary
    print('='*70)
    print('EVENT ALIGNMENT SUMMARY')
    print('='*70)
    print(f'Sync formula: Sample_Index = (Time_ms - {FORMULA_CONSTANT}) / {MS_PER_SAMPLE}')
    print(f'Neural data samples: {n_samples}')
    print(f'Sampling rate: {sampling_rate} Hz')
    print(f'Recording duration: {recording_duration:.2f} seconds')
    print()
    print(f'Total events: {len(events_aligned)}')
    print(f'Valid events (within recording): {events_aligned["Valid"].sum()}')
    print(f'Invalid events (outside recording): {(~events_aligned["Valid"]).sum()}')
    print()

    if events_aligned['Valid'].sum() > 0:
        print('Valid event range:')
        valid_events = events_aligned[events_aligned['Valid']]
        print(f'  First event: Time_ms {valid_events["Time_ms"].min()} → Sample {valid_events["Sample_Index"].min()}')
        print(f'  Last event:  Time_ms {valid_events["Time_ms"].max()} → Sample {valid_events["Sample_Index"].max()}')

    if (~events_aligned['Valid']).sum() > 0:
        print('\nWarning: Some events fall outside neural recording!')
        invalid_events = events_aligned[~events_aligned['Valid']]
        before = (invalid_events['Sample_Index'] < 0).sum()
        after  = (invalid_events['Sample_Index'] >= n_samples).sum()
        print(f'  Before recording start: {before}')
        print(f'  After recording end: {after}')

    print('='*70)

    return events_aligned

In [16]:
# Sync anchor: neural sample index = (Time_ms - FORMULA_CONSTANT) / MS_PER_SAMPLE
# Time_Relative = sample index / SAMPLING_RATE
# This is equivalent to align_events_to_neural_data with:
#   neural_data_start_time = 0 (sample 0 = task start anchor)
#   sampling_rate = 250

events_data = align_events_to_neural_data(
    events_data,
    sampling_rate=SAMPLING_RATE,
    neural_data_start_time=0.0,   # anchor: sample 0 corresponds to first stim
    n_samples=n_samples
)
events_data

EVENT ALIGNMENT SUMMARY
Sync formula: Sample_Index = (Time_ms - 1199133) / 4.0
Neural data samples: 23750
Sampling rate: 250 Hz
Recording duration: 95.00 seconds

Total events: 286
Valid events (within recording): 0
Invalid events (outside recording): 286


  Before recording start: 286
  After recording end: 0


,Time,Time_ms,Event,Block,Trial_In_Block,Event_Type,Trial_Number,Sample_Index,Time_Relative,Valid
0,-1165.768,33367,6.0,NaN,NaN,Session Start,<NA>,-291442,-1165.768,False
1,-1131.700,67434,8.0,1.0,1.0,Main Trial Start,1,-282925,-1131.700,False
2,-1131.700,67434,1.0,1.0,1.0,Fixation Start,1,-282925,-1131.700,False
3,-1131.684,67448,2.0,1.0,1.0,Fixation End,1,-282921,-1131.684,False
4,-1130.688,68447,10.0,1.0,1.0,Stimulus Start,1,-282672,-1130.688,False
...,...,...,...,...,...,...,...,...,...,...
281,-972.024,227109,13.0,7.0,2.0,Choice End,14,-243006,-972.024,False
282,-972.024,227109,14.0,7.0,2.0,Feedback Start,14,-243006,-972.024,False
283,-970.516,228618,15.0,7.0,2.0,Feedback End,14,-242629,-970.516,False
284,-970.516,228618,9.0,7.0,2.0,Main Trial End,14,-242629,-970.516,False


### Importing the eprime logs

In [17]:
# Import E-Prime scores and extract per-trial answers
eprime_data = pd.read_csv(EPRIME_DATA_PATH)
cols = ['CollectResponse.ACC', 'CollectResponse.CRESP', 'CollectResponse.RESP']
# One answer row per (Block, Trial) combination = 14 trials
answers = (eprime_data.groupby(['Block', 'Trial'], sort=False)[cols]
           .first().reset_index())
# Add a sequential trial number matching Trial_Number in events_data
answers['Trial_Number'] = range(1, len(answers) + 1)
answers = answers.rename(columns={'CollectResponse.ACC': 'ACC',
                                   'CollectResponse.CRESP': 'CRESP',
                                   'CollectResponse.RESP': 'RESP'})
answers

,Block,Trial,ACC,CRESP,RESP,Trial_Number
0,1,1,1,34,34,1
1,1,2,1,25,25,2
2,2,1,1,141,141,3
3,2,2,1,532,532,4
4,3,1,1,5432,5432,5
5,3,2,0,5421,5432,6
6,4,1,0,4213,4231,7
7,4,2,0,4531,5341,8
8,5,1,1,5132,5132,9
9,5,2,0,2341,2314,10


### Add trial answers to event logs

In [18]:
def add_trial_answers(events_df, answers_df):
    """
    Add accuracy, correct response, and actual response to events DataFrame.

    Parameters:
    -----------
    events_df : pandas.DataFrame
        Events DataFrame with Trial_Number column
    answers_df : pandas.DataFrame
        DataFrame with columns: ACC, CRESP, RESP, Trial_Number

    Returns:
    --------
    events_df : pandas.DataFrame
        Events DataFrame with added columns: ACC, CRESP, RESP
    """

    events_with_answers = events_df.copy()

    # Initialize new columns
    events_with_answers['ACC']   = np.nan
    events_with_answers['CRESP'] = np.nan
    events_with_answers['RESP']  = np.nan

    # Get valid trial numbers
    valid_trials = events_with_answers['Trial_Number'].dropna().unique()
    valid_trials = sorted([int(t) for t in valid_trials])

    print(f'Valid trials in events: {len(valid_trials)}')
    print(f'Trials in answers: {len(answers_df)}')

    # Map answers to trials by Trial_Number
    ans_map = answers_df.set_index('Trial_Number')

    for trial_num in valid_trials:
        if trial_num in ans_map.index:
            acc   = ans_map.loc[trial_num, 'ACC']
            cresp = ans_map.loc[trial_num, 'CRESP']
            resp  = ans_map.loc[trial_num, 'RESP']

            trial_mask = events_with_answers['Trial_Number'] == trial_num
            events_with_answers.loc[trial_mask, 'ACC']   = acc
            events_with_answers.loc[trial_mask, 'CRESP'] = cresp
            events_with_answers.loc[trial_mask, 'RESP']  = resp
        else:
            print(f'Warning: No answer data for trial {trial_num}')

    # Convert to appropriate types
    events_with_answers['ACC']   = events_with_answers['ACC'].astype('Int64')
    events_with_answers['CRESP'] = events_with_answers['CRESP'].astype('Int64')
    events_with_answers['RESP']  = events_with_answers['RESP'].astype('Int64')

    print('\nAnswer data added successfully!')
    print(f'Correct trials: {events_with_answers[events_with_answers["ACC"] == 1]["Trial_Number"].nunique()}')
    print(f'Incorrect trials: {events_with_answers[events_with_answers["ACC"] == 0]["Trial_Number"].nunique()}')

    return events_with_answers

In [19]:
events_data = add_trial_answers(events_data, answers)
events_data

Valid trials in events: 14
Trials in answers: 14

Answer data added successfully!
Correct trials: 6
Incorrect trials: 8


,Time,Time_ms,Event,Block,Trial_In_Block,Event_Type,Trial_Number,Sample_Index,Time_Relative,Valid,ACC,CRESP,RESP
0,-1165.768,33367,6.0,NaN,NaN,Session Start,<NA>,-291442,-1165.768,False,<NA>,<NA>,<NA>
1,-1131.700,67434,8.0,1.0,1.0,Main Trial Start,1,-282925,-1131.700,False,1,34,34
2,-1131.700,67434,1.0,1.0,1.0,Fixation Start,1,-282925,-1131.700,False,1,34,34
3,-1131.684,67448,2.0,1.0,1.0,Fixation End,1,-282921,-1131.684,False,1,34,34
4,-1130.688,68447,10.0,1.0,1.0,Stimulus Start,1,-282672,-1130.688,False,1,34,34
...,...,...,...,...,...,...,...,...,...,...,...,...,...
281,-972.024,227109,13.0,7.0,2.0,Choice End,14,-243006,-972.024,False,0,1254,1245
282,-972.024,227109,14.0,7.0,2.0,Feedback Start,14,-243006,-972.024,False,0,1254,1245
283,-970.516,228618,15.0,7.0,2.0,Feedback End,14,-242629,-970.516,False,0,1254,1245
284,-970.516,228618,9.0,7.0,2.0,Main Trial End,14,-242629,-970.516,False,0,1254,1245


### Add number of digits in trials to event logs

In [20]:
# Count the number of Stimulus Start events per trial (each digit has one stimulus start/end pair)
events_data['Num_Digits'] = events_data.groupby('Trial_Number')['Event_Type'].transform(
    lambda x: (x == 'Stimulus Start').sum()
)

# Verify
print("Digit count by trial:")
print(events_data.groupby('Trial_Number')['Num_Digits'].first().value_counts().sort_index())

events_data

Digit count by trial:
Num_Digits
2.0     2
3.0     2
4.0    10
Name: count, dtype: int64


,Time,Time_ms,Event,Block,Trial_In_Block,Event_Type,Trial_Number,Sample_Index,Time_Relative,Valid,ACC,CRESP,RESP,Num_Digits
0,-1165.768,33367,6.0,NaN,NaN,Session Start,<NA>,-291442,-1165.768,False,<NA>,<NA>,<NA>,NaN
1,-1131.700,67434,8.0,1.0,1.0,Main Trial Start,1,-282925,-1131.700,False,1,34,34,2.0
2,-1131.700,67434,1.0,1.0,1.0,Fixation Start,1,-282925,-1131.700,False,1,34,34,2.0
3,-1131.684,67448,2.0,1.0,1.0,Fixation End,1,-282921,-1131.684,False,1,34,34,2.0
4,-1130.688,68447,10.0,1.0,1.0,Stimulus Start,1,-282672,-1130.688,False,1,34,34,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
281,-972.024,227109,13.0,7.0,2.0,Choice End,14,-243006,-972.024,False,0,1254,1245,4.0
282,-972.024,227109,14.0,7.0,2.0,Feedback Start,14,-243006,-972.024,False,0,1254,1245,4.0
283,-970.516,228618,15.0,7.0,2.0,Feedback End,14,-242629,-970.516,False,0,1254,1245,4.0
284,-970.516,228618,9.0,7.0,2.0,Main Trial End,14,-242629,-970.516,False,0,1254,1245,4.0


### Add window labels event logs
Indicating whether each event belongs to Stimulus or Choice window.

In [21]:
def add_window_labels(events_df):
    """
    Add a column indicating whether each event belongs to Stimulus or Choice window.
    
    Window definitions:
    - Stimulus Window: From first 'Stimulus Start' to last 'Stimulus End' in a trial
    - Choice Window: From 'Choice Start' to 'Choice End' in a trial
    - Outside windows get NaN
    """
    
    events_with_windows = events_df.copy()
    events_with_windows['Window'] = np.nan
    events_with_windows['Window'] = events_with_windows['Window'].astype(object)  # allow strings
    
    # Get valid trial numbers
    valid_trials = events_with_windows['Trial_Number'].dropna().unique()
    valid_trials = sorted([int(t) for t in valid_trials])
    
    print(f'Processing {len(valid_trials)} trials...')
    
    for trial_num in valid_trials:
        trial_mask = events_with_windows['Trial_Number'] == trial_num
        trial_events = events_with_windows[trial_mask].copy()
        
        # Stimulus window
        stim_starts = trial_events[trial_events['Event_Type'] == 'Stimulus Start']
        stim_ends   = trial_events[trial_events['Event_Type'] == 'Stimulus End']
        
        if len(stim_starts) > 0 and len(stim_ends) > 0:
            first_stim_idx = stim_starts.index[0]
            last_stim_idx  = stim_ends.index[-1]
            for idx in trial_events.index:
                if first_stim_idx <= idx <= last_stim_idx:
                    events_with_windows.at[idx, 'Window'] = 'Stimulus'
        
        # Choice window
        choice_starts = trial_events[trial_events['Event_Type'] == 'Choice Start']
        choice_ends   = trial_events[trial_events['Event_Type'] == 'Choice End']
        
        if len(choice_starts) > 0 and len(choice_ends) > 0:
            choice_start_idx = choice_starts.index[0]
            choice_end_idx   = choice_ends.index[0]
            for idx in trial_events.index:
                if choice_start_idx <= idx <= choice_end_idx:
                    events_with_windows.at[idx, 'Window'] = 'Choice'
    
    # Print summary
    print('\n' + '='*70)
    print('WINDOW LABELING SUMMARY')
    print('='*70)
    print('\nWindow counts:')
    print(events_with_windows['Window'].value_counts())
    
    print('\nEvents per window type:')
    for window in ['Stimulus', 'Choice']:
        if window in events_with_windows['Window'].values:
            count  = (events_with_windows['Window'] == window).sum()
            trials = events_with_windows[events_with_windows['Window'] == window]['Trial_Number'].nunique()
            print(f'  {window}: {count} events across {trials} trials')
    
    unlabeled = events_with_windows['Window'].isna().sum()
    print(f'  Unlabeled (outside windows): {unlabeled} events')
    print('='*70)
    
    return events_with_windows

In [22]:
# Apply the function
events_data = add_window_labels(events_data)

# Display sample results
print("\nSample of events with window labels:")
events_data

Processing 14 trials...

WINDOW LABELING SUMMARY

Window counts:
Window
Stimulus    172
Choice       28
Name: count, dtype: int64

Events per window type:
  Stimulus: 172 events across 14 trials
  Choice: 28 events across 14 trials
  Unlabeled (outside windows): 86 events

Sample of events with window labels:


,Time,Time_ms,Event,Block,Trial_In_Block,Event_Type,Trial_Number,Sample_Index,Time_Relative,Valid,ACC,CRESP,RESP,Num_Digits,Window
0,-1165.768,33367,6.0,NaN,NaN,Session Start,<NA>,-291442,-1165.768,False,<NA>,<NA>,<NA>,NaN,NaN
1,-1131.700,67434,8.0,1.0,1.0,Main Trial Start,1,-282925,-1131.700,False,1,34,34,2.0,NaN
2,-1131.700,67434,1.0,1.0,1.0,Fixation Start,1,-282925,-1131.700,False,1,34,34,2.0,NaN
3,-1131.684,67448,2.0,1.0,1.0,Fixation End,1,-282921,-1131.684,False,1,34,34,2.0,NaN
4,-1130.688,68447,10.0,1.0,1.0,Stimulus Start,1,-282672,-1130.688,False,1,34,34,2.0,Stimulus
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
281,-972.024,227109,13.0,7.0,2.0,Choice End,14,-243006,-972.024,False,0,1254,1245,4.0,Choice
282,-972.024,227109,14.0,7.0,2.0,Feedback Start,14,-243006,-972.024,False,0,1254,1245,4.0,NaN
283,-970.516,228618,15.0,7.0,2.0,Feedback End,14,-242629,-970.516,False,0,1254,1245,4.0,NaN
284,-970.516,228618,9.0,7.0,2.0,Main Trial End,14,-242629,-970.516,False,0,1254,1245,4.0,NaN


## Saving the events data

In [23]:
events_data.to_csv("C:\\Users\\ASSUS\\ATN\\Digit Span Backwards\\Session 2\\Events.csv",index=False)